![](img/logo.png)


# Density Estimation with Normalizing Flows

Density estimation asks for a model of an unknown distribution $p(x)$ from samples.
In this notebook we keep the classical material brief and focus on a modern generative model:
a **normalizing flow** implemented in **JAX** with **FlowJax**.

We will use one dataset throughout:

1. fit compact KDE and GMM baselines on `make_moons`
2. show why these baselines struggle with curved support
3. train a masked autoregressive flow (MAF)
4. compare densities and generated samples


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KernelDensity

import flowjax.bijections
import flowjax.distributions
import flowjax.flows
import flowjax.train
import jax.numpy as jnp
from jax import random

plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(0)


In [ ]:

def make_grid(data, pad=0.6, n=220):
    lower = data.min(axis=0) - pad
    upper = data.max(axis=0) + pad
    xs = np.linspace(lower[0], upper[0], n)
    ys = np.linspace(lower[1], upper[1], n)
    xx, yy = np.meshgrid(xs, ys)
    grid = np.column_stack([xx.ravel(), yy.ravel()]).astype(np.float32)
    return xs, ys, grid, lower, upper


def plot_density(ax, xs, ys, log_density, data=None, title="", cmap="viridis"):
    density = np.exp(np.asarray(log_density)).reshape(len(ys), len(xs))
    ax.contourf(xs, ys, density, levels=30, cmap=cmap)
    if data is not None:
        ax.scatter(data[:, 0], data[:, 1], s=4, c="white", alpha=0.18, edgecolors="none")
    ax.set(title=title, xlabel="$x_1$", ylabel="$x_2$")
    ax.set_aspect("equal")


def clip_to_window(samples, lower, upper):
    mask = np.all((samples >= lower) & (samples <= upper), axis=1)
    return samples[mask]



## A Dataset That Exposes Model Bias

The two-moons dataset is useful because the true distribution is clearly **non-Gaussian** and lies on a curved manifold.
That makes it a good stress test for density models with different inductive biases.


In [ ]:

X, labels = make_moons(n_samples=2500, noise=0.06, random_state=0)
X = X.astype(np.float32)
xs, ys, grid, lower, upper = make_grid(X)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X[:, 0], X[:, 1], c=labels, s=6, cmap="viridis")
ax.set(title="Training data", xlabel="$x_1$", ylabel="$x_2$")
ax.set_aspect("equal")



## Compact Baselines: KDE and GMM

We keep two classical baselines:

1. **KDE** is flexible and local, but its smoothness is controlled by a bandwidth parameter.
2. **GMM** is parametric and easy to sample from, but it prefers elliptic clusters.

Both models are useful reference points before we train a flow.


In [ ]:

kde = KernelDensity(kernel="gaussian", bandwidth=0.18)
kde.fit(X)

gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=0)
gmm.fit(X)

kde_log_density = kde.score_samples(grid)
gmm_log_density = gmm.score_samples(grid)


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)

axes[0].scatter(X[:, 0], X[:, 1], c=labels, s=6, cmap="viridis")
axes[0].set(title="Data", xlabel="$x_1$", ylabel="$x_2$")
axes[0].set_aspect("equal")

plot_density(axes[1], xs, ys, kde_log_density, data=X, title="KDE")
plot_density(axes[2], xs, ys, gmm_log_density, data=X, title="GMM")



The KDE tracks the moons reasonably well but has no global latent representation and can oversmooth or leak mass between nearby regions.
The GMM is efficient and interpretable, but a small number of Gaussian components cannot bend naturally around the two crescents.

This is the point where normalizing flows become interesting: they keep **sampling** and **density evaluation**, but with a much more flexible geometry.



## Normalizing Flows

A normalizing flow starts from a simple latent distribution, usually $z \\sim \\mathcal{N}(0, I)$,
and learns an invertible transformation $x = f(z)$.
Because the map is invertible, we can both sample from the model and evaluate densities through the change-of-variables formula:

$$
\\log p_X(x) = \\log p_Z\\left(f^{-1}(x)\\right) + \\log \\left|\\det J_{f^{-1}}(x)\\right|.
$$

Here we use a **masked autoregressive flow** with **rational quadratic spline** transformations.
That keeps the model expressive enough for curved densities while still allowing tractable likelihood training.


In [ ]:

key = random.key(0)

key, subkey = random.split(key)
flow = flowjax.flows.masked_autoregressive_flow(
    subkey,
    base_dist=flowjax.distributions.Normal(jnp.zeros(X.shape[1])),
    transformer=flowjax.bijections.RationalQuadraticSpline(knots=8, interval=4),
    flow_layers=6,
    nn_width=64,
    nn_depth=2,
)

key, subkey = random.split(key)
flow, losses = flowjax.train.fit_to_data(
    subkey,
    flow,
    X,
    learning_rate=5e-4,
    batch_size=256,
    max_patience=15,
    max_epochs=80,
    show_progress=False,
)


In [ ]:

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(losses["train"], label="train")
ax.plot(losses["val"], label="validation")
ax.set(title="Flow training", xlabel="Epoch", ylabel="Negative log likelihood")
ax.legend()



The training and validation curves should decrease together if the flow is fitting the dataset without obvious instability.
Now we can compare all three fitted densities on the same grid.


In [ ]:

flow_log_density = np.asarray(flow.log_prob(jnp.asarray(grid)))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
plot_density(axes[0], xs, ys, kde_log_density, data=X, title="KDE")
plot_density(axes[1], xs, ys, gmm_log_density, data=X, title="GMM")
plot_density(axes[2], xs, ys, flow_log_density, data=X, title="Normalizing flow")



The difference is structural.
A GMM approximates the moons by combining ellipses.
A flow starts from a simple base density and learns a nonlinear invertible warp of space, so it can place probability mass along curved shapes while still remaining a normalized density model.


In [ ]:

kde_samples = clip_to_window(kde.sample(1500, random_state=0), lower, upper)
gmm_samples, _ = gmm.sample(1500)
gmm_samples = clip_to_window(gmm_samples, lower, upper)

key, subkey = random.split(key)
flow_samples = np.asarray(flow.sample(subkey, (1500,)))
flow_samples = clip_to_window(flow_samples, lower, upper)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4), constrained_layout=True)
axes[0].scatter(X[:, 0], X[:, 1], c=labels, s=6, cmap="viridis")
axes[0].set(title="Data", xlabel="$x_1$", ylabel="$x_2$")
axes[0].set_aspect("equal")

for ax, samples, title in zip(
    axes[1:],
    [kde_samples, gmm_samples, flow_samples],
    ["KDE samples", "GMM samples", "Flow samples"],
):
    ax.scatter(samples[:, 0], samples[:, 1], s=6, alpha=0.45)
    ax.set(title=title, xlabel="$x_1$", ylabel="$x_2$")
    ax.set_aspect("equal")



## Takeaways

1. KDE is a useful nonparametric baseline, but it is mainly a local smoother.
2. GMMs are simple and effective when cluster geometry is close to Gaussian.
3. Normalizing flows preserve exact density evaluation and direct sampling while learning much richer shapes.
4. JAX and FlowJax make it possible to express and train these models with relatively little code.

### Exercises

1. Change `flow_layers`, `nn_width`, or the number of spline `knots` and compare samples.
2. Increase or decrease the KDE bandwidth and compare the density heatmap.
3. Replace `make_moons` with another 2D dataset such as circles or blobs.
4. Try a larger GMM and compare whether it starts matching the moons at the cost of interpretability.



## References

1. [FlowJax documentation](https://danielward27.github.io/flowjax/)
2. [Masked Autoregressive Flow](https://arxiv.org/abs/1705.07057)
3. Kevin P. Murphy, *Probabilistic Machine Learning: An Introduction*
